In [1]:
import joblib
import json
import shap
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import japanize_matplotlib
from typing import Tuple, Any
from sklearn.model_selection import StratifiedKFold

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from src.data import load_train, load_test, convert_category
from src.config import load_config
from src.cv_features import create_cv_features

In [10]:
def calculate_shap(config_name: str, exp_dir_name: str) -> Tuple[np.ndarray, pd.DataFrame, pd.Series, pd.Series]:
    
    config = load_config(f"../configs/{config_name}.yaml")
    exp_name = config["experiment"]["name"]
        
        # モデルと特徴量、メトリクス、OOF予測値の読み込み
    model = joblib.load(f"../outputs/{exp_dir_name}/model.pkl")
    features = joblib.load(f"../outputs/{exp_dir_name}/feature_columns.pkl")
        
    with open(f"../outputs/{exp_dir_name}/metrics.json", "r", encoding="utf-8") as f:
        metrics = json.load(f)
            
    oof = pd.read_csv(f"../outputs/{exp_dir_name}/oof.csv")
        
        # データのロードと前処理
    train = load_train()
    test = load_test()
        
    target = config["data"]["target"]
    id_col = config["data"]["id"]
    drop_cols = config["feature"]["drop_columns"] + [target] + [id_col]
    cat_features = config["feature"]["categorical_features"]
        
    train, test = convert_category(train, test, cat_features)
        
    X = train.drop(columns=drop_cols)
    y = train[target]
    X = X[features]

    lgb_model = model.models[0]
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(X)

    return shap_values, X, y, oof

In [13]:
def run_shap_analysis(config_name: str, exp_dir_name: str) -> None:
    # 設定ファイルの読み込み
    config = load_config(f"../configs/{config_name}.yaml")
    exp_name = config["experiment"]["name"]
    
    # モデルと特徴量、メトリクス、OOF予測値の読み込み
    model = joblib.load(f"../outputs/{exp_dir_name}/model.pkl")
    features = joblib.load(f"../outputs/{exp_dir_name}/feature_columns.pkl")
    
    with open(f"../outputs/{exp_dir_name}/metrics.json", "r", encoding="utf-8") as f:
        metrics = json.load(f)
        
    oof = pd.read_csv(f"../outputs/{exp_dir_name}/oof.csv")
    
    # データのロードと前処理
    train = load_train()
    test = load_test()
    
    target = config["data"]["target"]
    id_col = config["data"]["id"]
    drop_cols = config["feature"]["drop_columns"] + [target] + [id_col]
    cat_features = config["feature"]["categorical_features"]
    
    train, test = convert_category(train, test, cat_features)
    
    X = train.drop(columns=drop_cols)
    y = train[target]
    X = X[features]
    
    # SHAP値の計算
    lgb_model = model.models[0]
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(X)
    
    # 個別のドットプロット生成・保存
    shap.summary_plot(shap_values, X, show=False)
    plt.savefig(f'images/{exp_name}_shap.png')
    plt.close()
    
    shap.summary_plot(shap_values[y == 1], X[y == 1], show=False)
    plt.savefig(f'images/{exp_name}_購入企業_shap.png')
    plt.close()
    
    # 見逃し (False Negative) の抽出とプロット
    fn = (y == 1) & (oof["prediction"] < metrics["best_threshold"])
    
    shap.summary_plot(shap_values[fn], X[fn], show=False)
    plt.savefig(f'images/{exp_name}_FN_shap.png')
    plt.close()
    
    # 誤検出　　(False Positive)　　の抽出
    fp = (y==0) & (oof["prediction"] > metrics["best_threshold"])

    shap.summary_plot(shap_values[fp], X[fp], show=False)
    plt.savefig(f'images/{exp_name}_FP_shap.png')
    plt.close()


    # バープロット比較図の生成・保存　
    fig, axes = plt.subplots(1, 4, figsize=(20, 15))
    
    plt.sca(axes[0])
    shap.summary_plot(shap_values, X, plot_type="bar", show=False)
    axes[0].set_title("All Data")
    
    plt.sca(axes[1])
    shap.summary_plot(shap_values[y == 1], X[y == 1], plot_type="bar", show=False)
    axes[1].set_title("y == 1")
    
    plt.sca(axes[2])
    shap.summary_plot(shap_values[fn], X[fn], plot_type="bar", show=False)
    axes[2].set_title("False Negative")

    plt.sca(axes[3])
    shap.summary_plot(shap_values[fp], X[fp], plot_type="bar", show=False)
    axes[3].set_title("False_Positive")

    plt.subplots_adjust(left=0.25, wspace=0.4)
    plt.savefig(f'images/{exp_name}_特徴量重要度比較.png')
    plt.close()

In [4]:
def run_oof_shap_analysis(config_name: str, exp_dir_name: str) -> None:
    # 1. 設定ファイルと学習済み成果物のロード
    config = load_config(f"../configs/{config_name}.yaml")
    exp_name = config["experiment"]["name"]

    model = joblib.load(f"../outputs/{exp_dir_name}/model.pkl")
    features = joblib.load(
        f"../outputs/{exp_dir_name}/feature_columns.pkl"
    )  # 学習時の79列

    with open(
        f"../outputs/{exp_dir_name}/metrics.json", "r", encoding="utf-8"
    ) as f:
        metrics = json.load(f)

    oof = pd.read_csv(f"../outputs/{exp_dir_name}/oof.csv")

    # 2. データのロードと前処理の設定
    train = load_train()
    target = config["data"]["target"]
    id_col = config["data"]["id"]
    cat_features = config["feature"]["categorical_features"]
    
    # 学習時と同じ不要カラムのリストを作成
    drop_cols = config["feature"]["drop_columns"] + [target] + [id_col]

    # 1) カテゴリ変換を行う
    train, _ = convert_category(train, train, cat_features)

    # 2) 学習時（train_cv.py）と同様に drop_cols をすべて削除して X_raw を作成
    X_raw = train.drop(columns=drop_cols)
    y_raw = train[target]

    n_splits = config.get("train", {}).get("n_splits", 5)
    random_state = config.get("train", {}).get("random_state", 42)
    skf = StratifiedKFold(
        n_splits=n_splits, shuffle=True, random_state=random_state
    )

    oof_shap_list = []
    oof_X_list = []
    oof_y_list = []
    oof_idx_list = []

    # 3. 各 Fold ごとに Validation データに対する SHAP を計算
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_raw, y_raw)):
        X_tr_raw = X_raw.iloc[train_idx].copy()
        X_val_raw = X_raw.iloc[val_idx].copy()
        y_val = y_raw.iloc[val_idx].copy()

        # --- 特徴量生成 (学習時と同じ入力状態で実行) ---
        X_tr, stats_dict = create_cv_features(X_tr_raw)
        X_val, _ = create_cv_features(X_val_raw, stats_dict=stats_dict)

        # --- 最終調整: 保存しておいた学習時カラム (79列) だけを抽出し、順序を一致させる ---
        X_val = X_val[features]

        # 対応する Fold のモデルで SHAP 値を計算
        fold_model = model.models[fold]
        explainer = shap.TreeExplainer(fold_model)
        shap_vals = explainer.shap_values(X_val)

        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]

        oof_shap_list.append(shap_vals)
        oof_X_list.append(X_val)
        oof_y_list.append(y_val)
        oof_idx_list.append(val_idx)

    # 4. 全 Fold の結果を結合し、元データのインデックス順に整列
    all_indices = np.concatenate(oof_idx_list)
    reorder_idx = np.argsort(all_indices)

    shap_values = np.vstack(oof_shap_list)[reorder_idx]
    X = pd.concat(oof_X_list, axis=0).iloc[reorder_idx].reset_index(drop=True)
    y = pd.concat(oof_y_list, axis=0).iloc[reorder_idx].reset_index(drop=True)

    # 5. 各種 SHAP プロットの生成と保存
    # 個別のドットプロット生成・保存
    shap.summary_plot(shap_values, X, show=False)
    plt.savefig(f"images/{exp_name}_shap.png")
    plt.close()

    shap.summary_plot(shap_values[y == 1], X[y == 1], show=False)
    plt.savefig(f"images/{exp_name}_購入企業_shap.png")
    plt.close()

    # 見逃し (False Negative) の抽出とプロット
    fn = (y == 1) & (oof["prediction"] < metrics["best_threshold"])

    shap.summary_plot(shap_values[fn], X[fn], show=False)
    plt.savefig(f"images/{exp_name}_FN_shap.png")
    plt.close()

    # 誤検出 (False Positive) の抽出
    fp = (y == 0) & (oof["prediction"] > metrics["best_threshold"])

    shap.summary_plot(shap_values[fp], X[fp], show=False)
    plt.savefig(f"images/{exp_name}_FP_shap.png")
    plt.close()

    # バープロット比較図の生成・保存
    fig, axes = plt.subplots(1, 4, figsize=(20, 15))

    plt.sca(axes[0])
    shap.summary_plot(shap_values, X, plot_type="bar", show=False)
    axes[0].set_title("All Data")

    plt.sca(axes[1])
    shap.summary_plot(
        shap_values[y == 1], X[y == 1], plot_type="bar", show=False
    )
    axes[1].set_title("y == 1")

    plt.sca(axes[2])
    shap.summary_plot(shap_values[fn], X[fn], plot_type="bar", show=False)
    axes[2].set_title("False Negative")

    plt.sca(axes[3])
    shap.summary_plot(shap_values[fp], X[fp], plot_type="bar", show=False)
    axes[3].set_title("False_Positive")

    plt.subplots_adjust(left=0.25, wspace=0.4)
    plt.savefig(f"images/{exp_name}_特徴量重要度比較.png")
    plt.close()

### exp001

In [ ]:
run_shap_analysis('baseline', 'exp001')

### exp002

In [17]:
run_shap_analysis('exp002', 'exp002')

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()


In [19]:
shap_values, X, y, oof = calculate_shap("exp002", "exp002")

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [36]:
cols = [
    "sw支出_対_売上比",
    "sw支出_対_総資産比",
    "売上高営業利益率",
    "売上高経常利益率"
]

# 2行2列の図領域を作成 (figsizeは全体サイズに合わせて適宜調整)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# axes を1次元配列に平坦化してループ処理を扱いやすくする
axes_flat = axes.flatten()

for i, col in enumerate(cols):
    plt.sca(axes_flat[i]) # カレントの描画先を指定
    shap.dependence_plot(
        col,
        shap_values,
        X,
        interaction_index="業界",
        ax=axes_flat[i],  # 描画対象の Subplot を渡す
        show=False        # 自動表示・破棄を防止
    )

# レイアウトを整えて保存
plt.tight_layout()
plt.savefig("images/exp002_相互shap_業界.png", dpi=300, bbox_inches="tight")
plt.close() # メモリ解放

In [39]:
cols = [
    "アンケート２",
    "アンケート４",
    "アンケート７",
    "アンケート８",
    "アンケート１０",
]

# 2行2列の図領域を作成 (figsizeは全体サイズに合わせて適宜調整)
fig, axes = plt.subplots(2, 3, figsize=(16, 12))

# axes を1次元配列に平坦化してループ処理を扱いやすくする
axes_flat = axes.flatten()

for i, col in enumerate(cols):
    plt.sca(axes_flat[i]) # カレントの描画先を指定
    shap.dependence_plot(
        "sw支出_対_売上比",
        shap_values,
        X,
        interaction_index=col,
        ax=axes_flat[i],  # 描画対象の Subplot を渡す
        show=False        # 自動表示・破棄を防止
    )

# レイアウトを整えて保存
plt.tight_layout()
plt.savefig("images/exp002_相互shap_sw支出_アンケート.png", dpi=300, bbox_inches="tight")
plt.close() # メモリ解放

### exp003

In [6]:
run_shap_analysis('exp003', 'exp003')

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()


### exp004

In [6]:
run_shap_analysis('exp004', 'exp004')

/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()
/opt/anaconda3/envs/StudentCup26env/lib/python3.11/site-packages/shap/plots/_beeswarm.py:1150: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  plt.tight_layout()


### exp005

In [5]:
run_oof_shap_analysis('exp005', 'exp005')

[LightGBM] [Fatal] The number of features in data (67) is not the same as it was in training data (79).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.


LightGBMError: The number of features in data (67) is not the same as it was in training data (79).
You can set ``predict_disable_shape_check=true`` to discard this error, but please be aware what you are doing.